In [3]:
# diff_jsonl.py
import json
import sys

file2 = "/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/evaluation/compbench/text/run_20261012_100342_273591/results/results.jsonl"
file1 = "/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/run_20261007_000000_000000/compbench_local_multiobject_mixed.jsonl"

with open(file2) as f:
    ids2 = {
        json.loads(line)["sample_id"]
        for line in f if line.strip()
    }

missing = []

with open(file1) as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            if row["sample_id"] not in ids2:
                missing.append(row)

print(f"Missing rows: {len(missing)}", file=sys.stderr)

for row in missing:
    print(json.dumps(row, ensure_ascii=False))

{"sample_id": "replace:2a2745e5/9/00009.png", "benchmark": "CompBench", "split": "train", "original_instruction": "replace the white horse on the right with a brown horse", "mixed_instruction": "replace <v1> with a brown horse", "references": [{"token": "<v1>", "entity_id": "e1", "span": "the white horse on the right", "start": 8, "end": 36, "role": "target", "grounding_query": "the white horse on the right", "region": {"bbox_xyxy": [346.0, 244.0, 557.0, 413.0], "image_size": [640, 640], "confidence": 1.0, "label": "the white horse on the right", "source": "benchmark_mask:edit_region", "crop_path": "assets/compbench_local_multiobject_mixed/replace_2a2745e5_9_00009.png/e1_crop.png", "mask_path": "assets/compbench_local_multiobject_mixed/replace_2a2745e5_9_00009.png/e1_mask.png"}, "visual_prompts": {"geometry_source": "mask", "coordinate_space": "pixel", "image_size": [640, 640], "box": {"xyxy": [346, 244, 557, 413], "normalized_xyxy": [0.540625, 0.38125, 0.8703125, 0.6453125]}, "point":

Missing rows: 3


In [7]:
import json
import sys

list_file = "/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/list.txt"
jsonl_file = "/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/run_20261007_000000_000000/compbench_local_multiobject_mixed.jsonl"
output_file = "/Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/temp.jsonl"



# Read list.txt in its existing order
with open(list_file) as f:
    filenames = [line.strip() for line in f if line.strip()]

# Remove the numeric prefix, e.g. "002297_"
order = {
    name.split("_", 1)[1]: i
    for i, name in enumerate(filenames)
}

# Read JSONL
records = []
with open(jsonl_file) as f:
    for line in f:
        if line.strip():
            obj = json.loads(line)

            # Convert sample_id to filename format
            key = obj["sample_id"].replace(":", "_").replace("/", "_")

            records.append((key, line))

# Check for unmatched entries
unmatched = [key for key, _ in records if key not in order]
missing = set(order) - {key for key, _ in records}

print(f"List entries: {len(filenames)}")
print(f"JSONL entries: {len(records)}")
print(f"Unmatched JSONL entries: {len(unmatched)}")
print(f"Missing JSONL entries: {len(missing)}")

if unmatched:
    print("Unmatched examples:", unmatched[:5])

if missing:
    print("Missing examples:", sorted(missing)[:5])

# Stop rather than silently produce incorrectly ordered output
if unmatched or missing:
    sys.exit("Error: list.txt and JSONL do not fully match.")

# Sort according to list.txt
records.sort(key=lambda item: order[item[0]])

# Save
with open(output_file, "w") as f:
    for _, line in records:
        f.write(line.rstrip("\n") + "\n")

print(f"Saved sorted JSONL to {output_file}")

List entries: 2609
JSONL entries: 2609
Unmatched JSONL entries: 0
Missing JSONL entries: 0
Saved sorted JSONL to /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/temp.jsonl


# CompBench Local + Multi-object → mixed-modality prompts

Loads the official `BohanJia/CompBench` `train` split and selects the Local Editing (`add`, `remove`, `replace`) and Multi-object Editing (`multi_object_add`, `multi_object_remove`) tasks used by this project. Set `RESUME_LAST_RUN = True` to continue the latest run: saved sample IDs are skipped and every missing sample is retried, even if it is not in the audit file. Add IDs to `RETRY_SAMPLE_IDS` to replace specific accepted annotations; otherwise a new timestamped run folder is created.

Install once from the repository root: `uv sync`. Copy `.env.example` to `.env` and set `GEMINI_API_KEY`. CompBench references are grounded only by the dataset-provided masks; no object-grounding model is used. Change `COMPBENCH_TASKS` if you want a different task subset.

In [2]:
import json

from annotation import (
    AnnotationConfig,
    compbench,
    comma_separated_env,
    create_annotation_run_directory,
    resolve_annotation_run_directory,
    run_hf_annotation,
)

DATASET_ID = "BohanJia/CompBench"
SPLIT = "train"
TASKS = comma_separated_env(
    "COMPBENCH_TASKS",
    "add,remove,replace,multi_object_add,multi_object_remove"
)
CONFIG = AnnotationConfig.from_env("compbench")
RESUME_LAST_RUN = True  # Resume latest run: process only sample IDs absent from output.
RETRY_FAILED_ONLY = True  # Retained compatibility; also resumes missing IDs.
RETRY_SAMPLE_IDS = set()  # Exact CompBench sample IDs to replace, e.g. {'multi_object_remove:path/to/image.png'}.
print("tasks:", sorted(TASKS))

tasks: ['add', 'multi_object_add', 'multi_object_remove', 'remove', 'replace']


In [3]:
from dataclasses import replace

TARGETED_RETRY = RESUME_LAST_RUN or RETRY_FAILED_ONLY or bool(RETRY_SAMPLE_IDS)
RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN or TARGETED_RETRY
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(
    CONFIG,
    output_dir=RUN_DIR,
    limit=None if TARGETED_RETRY else CONFIG.limit,
)
print("Annotation run directory:", RUN_DIR)

AUDIT_PATH = RUN_DIR / "compbench_local_multiobject_audit.jsonl"
ANNOTATIONS_PATH = RUN_DIR / "compbench_local_multiobject_mixed.jsonl"

def read_sample_ids(path, statuses=None):
    if not path.exists():
        return set()
    with path.open(encoding="utf-8") as stream:
        return {
            record["sample_id"]
            for line in stream
            if line.strip()
            for record in [json.loads(line)]
            if record.get("sample_id")
            and (statuses is None or record.get("status") in statuses)
        }

ACCEPTED_SAMPLE_IDS = read_sample_ids(ANNOTATIONS_PATH)
FAILED_SAMPLE_IDS = (
    read_sample_ids(AUDIT_PATH, {"failed", "skipped"}) - ACCEPTED_SAMPLE_IDS
)
print("Accepted annotations loaded:", len(ACCEPTED_SAMPLE_IDS))
print("Unresolved audited samples:", len(FAILED_SAMPLE_IDS))
print("All missing dataset samples will be selected for retry.")

summary = run_hf_annotation(
    dataset_id=DATASET_ID,
    split=SPLIT,
    adapter=compbench,
    config=RUN_CONFIG,
    output_name="compbench_local_multiobject_mixed.jsonl",
    audit_name="compbench_local_multiobject_audit.jsonl",
    expected_columns={"task", "image_path", "input_image", "edited_image", "instruction", "caption", "mask"},
    use_grounder=False,
    overwrite_sample_ids=RETRY_SAMPLE_IDS,
    row_filter=lambda row: row["task"] in TASKS and (
        not TARGETED_RETRY
        or f"{row['task']}:{row['image_path']}" not in ACCEPTED_SAMPLE_IDS
        or f"{row['task']}:{row['image_path']}" in RETRY_SAMPLE_IDS
    ),
)
summary

Annotation run directory: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/compbench/run_20261007_000000_000000
Accepted annotations loaded: 2590
Unresolved audited samples: 19
All missing dataset samples will be selected for retry.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


train: 0it [00:00, ?it/s]

{'accepted': 19, 'skipped': 0, 'failed': 0, 'resumed': 0, 'retried': 19}

CompBench masks are nullable only for `action`, `location`, and `view`; the selected Local/Multi-object tasks therefore normally carry edit-region masks. All references in this notebook are parsed and grounded from the dataset-provided mask; no text-based object-grounding model is used. `add` uses its edit mask as a spatial destination reference. For multi-object add/remove, connected components and under-segmented masks are split into spatial partitions using projection, distance-watershed, and spatial-clustering methods, then parsed independently. Multi-object remove preserves separate masks even when the parser maps them to the same collective phrase. Resuming compares dataset IDs with saved annotations and retries every missing row, whether or not it appears in the audit file; successful targeted re-annotations replace only selected existing rows.